# Validacao da mart de indicadores financeiros

Este notebook valida a tabela `layer_03_gold.mart_indicadores_financeiros` apos a execucao da query da camada gold.


## 1. Setup


In [1]:
from __future__ import annotations

import os
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'README.md').exists() and (candidate / 'queries').exists():
            return candidate
    raise FileNotFoundError('Nao foi possivel localizar a raiz do repositorio.')


ROOT = find_repo_root()
load_dotenv(ROOT / '.env')

DB_USER = quote_plus(os.getenv('DB_USER', 'postgres'))
DB_PASS = quote_plus(os.getenv('DB_PASS', 'password'))
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'data_lake')

engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

print(f'Banco alvo: {DB_HOST}:{DB_PORT}/{DB_NAME}')


Banco alvo: localhost:5433/bigdata_for_finance


## 2. Carregar a mart


In [2]:
q_mart = '''
SELECT *
FROM "layer_03_gold"."mart_indicadores_financeiros";
'''

df = pd.read_sql(text(q_mart), engine)
df['DT_REFER'] = pd.to_datetime(df['DT_REFER'])

print(f'Linhas carregadas: {len(df):,}')
print(f'Empresas unicas: {df["CNPJ_CIA"].nunique():,}')
print(f'Setores unicos: {df["SETOR"].nunique(dropna=True):,}')
display(df.head())


Linhas carregadas: 2,221
Empresas unicas: 185
Setores unicos: 23


,CNPJ_CIA,DT_REFER,RAZAO_SOCIAL,SETOR,TP_MERC,AT,AC,CAIXA_EQUIV,APLIC_FIN,CLIENTES,...,GIRO_AC,NCG,ST,PMRV,PMRE,PMPC,PMRAC,CICLO_ECONOMICO,CICLO_FINANCEIRO,EFEITO_TESOURA
0,00.070.698/0001-11,2010-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,2.119934e+09,449960000.0,99258000.0,NaN,321170000.0,...,4.087741,-123045000.0,-54941000.0,62.860840,4.823122,80.014363,88.068199,67.683962,-12.330401,False
1,00.070.698/0001-11,2011-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,2.170285e+09,457284000.0,66748000.0,NaN,357186000.0,...,3.012611,-139630000.0,-60851000.0,93.340002,3.065170,52.313517,119.497655,96.405172,44.091655,False
2,00.070.698/0001-11,2012-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,2.424419e+09,570535000.0,185433000.0,9805000.0,308111000.0,...,2.854650,-133982000.0,89225000.0,68.104291,2.359371,46.573824,126.110011,70.463662,23.889838,False
3,00.070.698/0001-11,2013-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,2.434831e+09,520802000.0,96786000.0,295000.0,308840000.0,...,3.088838,-354374000.0,-22126000.0,69.114357,8.964314,89.112503,116.548683,78.078671,-11.033833,False
4,00.070.698/0001-11,2014-12-31,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,MERCADO_ABERTO,2.709827e+09,779411000.0,66006000.0,NaN,441174000.0,...,2.723573,-164177000.0,-3411000.0,74.818123,3.848033,77.714648,132.179295,78.666156,0.951508,False


## 3. Sanidade basica


In [3]:
q_dupes = '''
SELECT "CNPJ_CIA", "DT_REFER", COUNT(*) AS n
FROM "layer_03_gold"."mart_indicadores_financeiros"
GROUP BY 1, 2
HAVING COUNT(*) > 1
ORDER BY n DESC, "CNPJ_CIA", "DT_REFER";
'''

df_dupes = pd.read_sql(text(q_dupes), engine)
print(f'Chaves duplicadas CNPJ_CIA + DT_REFER: {len(df_dupes)}')
display(df_dupes.head(20))


Chaves duplicadas CNPJ_CIA + DT_REFER: 0


,CNPJ_CIA,DT_REFER,n


## 4. Cobertura dos indicadores


In [4]:
indicator_cols = [
    'LIQ_GERAL', 'LIQ_CORRENTE', 'LIQ_SECA', 'LIQ_IMEDIATA',
    'ENDIV_CP', 'ENDIV_TOTAL', 'GARANTIA_CP_CT', 'COMPOSICAO_ENDIV', 'IMOB_PL', 'IMOB_AT',
    'MARGEM_BRUTA', 'MARGEM_OPERACIONAL', 'MARGEM_LIQUIDA', 'LPA',
    'ROA', 'ROE', 'ROI',
    'GIRO_ESTOQUES', 'GIRO_CLIENTES', 'GIRO_FORNECEDORES', 'GIRO_AC',
    'PMRE', 'PMRV', 'PMPC', 'PMRAC',
    'CICLO_ECONOMICO', 'CICLO_FINANCEIRO',
    'CGL', 'NCG', 'ST'
]

coverage = pd.DataFrame({
    'indicador': indicator_cols,
    'preenchidos': [int(df[c].notna().sum()) for c in indicator_cols],
})
coverage['cobertura_pct'] = (coverage['preenchidos'] / len(df) * 100).round(2)
coverage = coverage.sort_values(['cobertura_pct', 'indicador'], ascending=[False, True]).reset_index(drop=True)
display(coverage)


,indicador,preenchidos,cobertura_pct
0,CGL,2221,100.00
1,NCG,2221,100.00
2,ST,2221,100.00
3,GIRO_FORNECEDORES,2220,99.95
4,LIQ_CORRENTE,2220,99.95
5,LIQ_IMEDIATA,2220,99.95
6,LIQ_SECA,2220,99.95
7,ROA,2220,99.95
8,ROI,2220,99.95
9,COMPOSICAO_ENDIV,2218,99.86


## 5. Validacao de ativos biologicos

Checa se a mart esta incorporando ativos biologicos somente quando eles existem.


In [5]:
bio_summary = pd.DataFrame({
    'linhas_com_ativo_biologico': [int(df['AT_BIOLOGICO'].notna().sum())],
    'empresas_com_ativo_biologico': [int(df.loc[df['AT_BIOLOGICO'].notna(), 'CNPJ_CIA'].nunique())],
    'linhas_sem_ativo_biologico': [int(df['AT_BIOLOGICO'].isna().sum())],
})
display(bio_summary)

df_bio = df.loc[df['AT_BIOLOGICO'].notna(), [
    'CNPJ_CIA', 'RAZAO_SOCIAL', 'SETOR', 'DT_REFER', 'ESTOQUES', 'AT_BIOLOGICO',
    'LIQ_SECA', 'GIRO_ESTOQUES', 'PMRE', 'NCG'
]].sort_values(['SETOR', 'RAZAO_SOCIAL', 'DT_REFER'])

display(df_bio.head(30))


,linhas_com_ativo_biologico,empresas_com_ativo_biologico,linhas_sem_ativo_biologico
0,98,10,2123


,CNPJ_CIA,RAZAO_SOCIAL,SETOR,DT_REFER,ESTOQUES,AT_BIOLOGICO,LIQ_SECA,GIRO_ESTOQUES,PMRE,NCG
377,07.628.528/0001-59,BRASILAGRO - CIA BRAS DE PROP AGRICOLAS,"Agricultura (Açúcar, Álcool e Cana)",2011-06-30,77479000.0,1335000.0,1.534048,0.780318,461.350244,41046000.0
378,07.628.528/0001-59,BRASILAGRO - CIA BRAS DE PROP AGRICOLAS,"Agricultura (Açúcar, Álcool e Cana)",2012-06-30,72558000.0,4111000.0,1.286135,1.779689,202.282498,83972000.0
379,07.628.528/0001-59,BRASILAGRO - CIA BRAS DE PROP AGRICOLAS,"Agricultura (Açúcar, Álcool e Cana)",2013-06-30,28805000.0,1201000.0,2.108823,5.686963,63.302685,117006000.0
380,07.628.528/0001-59,BRASILAGRO - CIA BRAS DE PROP AGRICOLAS,"Agricultura (Açúcar, Álcool e Cana)",2014-06-30,40210000.0,1421000.0,1.094188,3.327689,108.183203,12518000.0
381,07.628.528/0001-59,BRASILAGRO - CIA BRAS DE PROP AGRICOLAS,"Agricultura (Açúcar, Álcool e Cana)",2015-06-30,32225000.0,1624000.0,2.085147,5.036751,71.474641,-48162000.0
383,07.628.528/0001-59,BRASILAGRO - CIA BRAS DE PROP AGRICOLAS,"Agricultura (Açúcar, Álcool e Cana)",2017-06-30,22658000.0,38260000.0,0.701112,2.238452,160.825450,19796000.0
384,07.628.528/0001-59,BRASILAGRO - CIA BRAS DE PROP AGRICOLAS,"Agricultura (Açúcar, Álcool e Cana)",2018-06-30,69622000.0,61993000.0,1.184644,1.734749,207.522808,123685000.0
385,07.628.528/0001-59,BRASILAGRO - CIA BRAS DE PROP AGRICOLAS,"Agricultura (Açúcar, Álcool e Cana)",2019-06-30,97068000.0,99881000.0,1.076163,1.620795,222.113191,180406000.0
386,07.628.528/0001-59,BRASILAGRO - CIA BRAS DE PROP AGRICOLAS,"Agricultura (Açúcar, Álcool e Cana)",2020-06-30,138778000.0,115553000.0,0.970109,1.902297,189.244936,314450000.0
387,07.628.528/0001-59,BRASILAGRO - CIA BRAS DE PROP AGRICOLAS,"Agricultura (Açúcar, Álcool e Cana)",2021-06-30,265859000.0,210489000.0,1.942951,1.530698,235.186801,393398000.0


## 6. Validacao setorial e COSAN


In [6]:
setor_alvo = 'Agricultura (Açúcar, Álcool e Cana)'
cnpj_cosan = '50.746.577/0001-15'

df_setor = df[df['SETOR'] == setor_alvo].copy()
df_cosan = df[df['CNPJ_CIA'] == cnpj_cosan].copy()

print(f'Linhas do setor alvo: {len(df_setor):,}')
print(f'Linhas da COSAN: {len(df_cosan):,}')

display(df_cosan[[
    'DT_REFER', 'RAZAO_SOCIAL', 'AT_BIOLOGICO', 'LIQ_SECA', 'GIRO_ESTOQUES', 'PMRE', 'NCG', 'ST'
]].sort_values('DT_REFER'))


Linhas do setor alvo: 86
Linhas da COSAN: 15


,DT_REFER,RAZAO_SOCIAL,AT_BIOLOGICO,LIQ_SECA,GIRO_ESTOQUES,PMRE,NCG,ST
1246,2011-03-31,COSAN S.A.,NaN,1.136140,22.600893,15.928574,6.672080e+08,3.376700e+08
1247,2012-03-31,COSAN S.A.,NaN,1.909097,28.690783,12.547584,1.558163e+09,1.079034e+09
1248,2013-03-31,COSAN S.A.,NaN,1.199832,29.261951,12.302666,1.463863e+09,3.893420e+08
1249,2013-12-31,COSAN S.A.,NaN,1.226525,15.636352,23.023273,4.224270e+08,4.869570e+08
1250,2014-12-31,COSAN S.A.,NaN,1.202449,16.680969,21.581481,1.793210e+08,6.838580e+08
1251,2015-12-31,COSAN S.A.,NaN,1.415435,13.739474,26.201876,-7.824200e+07,1.899012e+09
1252,2016-12-31,COSAN S.A.,NaN,1.848602,13.249306,27.171235,2.612200e+07,3.054930e+09
1253,2017-12-31,COSAN S.A.,NaN,1.482408,13.251790,27.166142,1.249876e+09,1.538781e+09
1254,2018-12-31,COSAN S.A.,NaN,1.786743,16.989000,21.190182,2.059583e+09,1.535744e+09
1255,2019-12-31,COSAN S.A.,NaN,1.751178,17.795210,20.230163,1.474753e+09,3.695862e+09


## 7. Comparativo COSAN vs mediana do setor


In [7]:
compare_cols = [
    'LIQ_CORRENTE', 'LIQ_SECA', 'MARGEM_LIQUIDA', 'ROE',
    'GIRO_ESTOQUES', 'PMRE', 'CICLO_FINANCEIRO', 'NCG', 'ST'
]

setor_med = (
    df_setor.assign(ANO=df_setor['DT_REFER'].dt.year)
    .groupby('ANO')[compare_cols]
    .median()
    .reset_index()
)

cosan = (
    df_cosan.assign(ANO=df_cosan['DT_REFER'].dt.year)
    [['ANO', *compare_cols]]
    .sort_values('ANO')
)

df_compare = cosan.merge(setor_med, on='ANO', suffixes=('_cosan', '_mediana_setor'))
display(df_compare)


,ANO,LIQ_CORRENTE_cosan,LIQ_SECA_cosan,MARGEM_LIQUIDA_cosan,ROE_cosan,GIRO_ESTOQUES_cosan,PMRE_cosan,CICLO_FINANCEIRO_cosan,NCG_cosan,ST_cosan,LIQ_CORRENTE_mediana_setor,LIQ_SECA_mediana_setor,MARGEM_LIQUIDA_mediana_setor,ROE_mediana_setor,GIRO_ESTOQUES_mediana_setor,PMRE_mediana_setor,CICLO_FINANCEIRO_mediana_setor,NCG_mediana_setor,ST_mediana_setor
0,2011,1.408923,1.136140,0.042991,0.114464,22.600893,15.928574,14.506366,6.672080e+08,3.376700e+08,1.399853,1.059410,0.127296,0.078078,3.693327,217.895248,125.304258,236446000.0,90900500.0
1,2012,2.269141,1.909097,0.109757,0.275030,28.690783,12.547584,16.779259,1.558163e+09,1.079034e+09,1.343121,1.016387,0.092620,0.062534,5.812560,61.934849,85.678227,83972000.0,24397000.0
2,2013,1.393425,1.199832,0.028928,0.064821,29.261951,12.302666,13.869290,1.463863e+09,3.893420e+08,1.369124,1.213178,0.060332,0.041960,6.992838,53.341452,59.307600,269716500.0,155385000.0
3,2013,1.344822,1.226525,0.076073,0.038715,15.636352,23.023273,3.577902,4.224270e+08,4.869570e+08,1.369124,1.213178,0.060332,0.041960,6.992838,53.341452,59.307600,269716500.0,155385000.0
4,2014,1.339139,1.202449,0.078962,0.051673,16.680969,21.581481,-2.321227,1.793210e+08,6.838580e+08,1.243075,1.094188,0.046788,0.029312,5.961206,60.390463,28.570815,56972000.0,46024000.0
5,2015,1.544317,1.415435,0.114562,0.077247,13.739474,26.201876,-35.328269,-7.824200e+07,1.899012e+09,1.245358,1.107236,0.114562,0.077247,6.027964,59.721653,2.001356,-5329000.0,147693000.0
6,2016,1.956016,1.848602,0.184318,0.128803,13.249306,27.171235,-53.962139,2.612200e+07,3.054930e+09,1.333302,1.144885,0.076020,0.044394,6.010530,59.894888,21.083516,26122000.0,116148000.0
7,2017,1.558693,1.482408,0.196920,0.146692,13.251790,27.166142,-58.536713,1.249876e+09,1.538781e+09,1.247993,0.701112,0.148232,0.110040,2.285120,157.540953,57.036647,699755000.0,-6108000.0
8,2018,1.900133,1.786743,0.184846,0.174286,16.989000,21.190182,-8.250990,2.059583e+09,1.535744e+09,1.832506,1.184644,0.143972,0.158304,2.549536,141.202179,130.455910,691572000.0,45441000.0
9,2019,1.838558,1.751178,0.183126,0.224503,17.795210,20.230163,-5.545538,1.474753e+09,3.695862e+09,1.891901,1.413670,0.154748,0.153333,3.632029,104.628608,88.188420,447286000.0,195063500.0


## 8. Alertas de qualidade


In [8]:
alerts = {
    'liq_corrente_negativa': int((df['LIQ_CORRENTE'] < 0).sum()),
    'pmre_negativo': int((df['PMRE'] < 0).sum()),
    'pmrv_negativo': int((df['PMRV'] < 0).sum()),
    'pmpc_negativo': int((df['PMPC'] < 0).sum()),
    'roe_com_pl_negativo': int(((df['PL'] < 0) & df['ROE'].notna()).sum()),
    'lpa_nulo': int(df['LPA'].isna().sum()),
}

display(pd.DataFrame([alerts]))


,liq_corrente_negativa,pmre_negativo,pmrv_negativo,pmpc_negativo,roe_com_pl_negativo,lpa_nulo
0,0,0,0,0,0,285
